# Indo Fashion Dataset Exploration

This notebook is an optional space for quick EDA before training.

## 1) Setup

Load standard libraries and define project paths.

In [ ]:
from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

PROJECT_ROOT = Path("..").resolve()
DATASET_ROOT = PROJECT_ROOT / "data" / "indofashion_dataset"
SUBSET_ROOT = PROJECT_ROOT / "data" / "subset"
OUTPUTS_ROOT = PROJECT_ROOT / "outputs"

CLASS_NAMES = [
    "blouse", "dhoti_pants", "dupattas", "gowns", "kurta_men",
    "lehenga", "mojaris_men", "mojaris_women", "nehru_jacket",
    "palazzo", "petticoats", "saree", "sherwanis", "women_kurta", "leggings_and_salwars"
]

print("Project root:", PROJECT_ROOT)
print("Dataset root exists:", DATASET_ROOT.exists())
print("Subset root exists:", SUBSET_ROOT.exists())

## 2) Load Annotation Files

The provided dataset uses `*_data.json` files in NDJSON format (one JSON object per line).

In [ ]:
def load_ndjson(path: Path) -> pd.DataFrame:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return pd.DataFrame(rows)

train_df = load_ndjson(DATASET_ROOT / "train_data.json")
val_df = load_ndjson(DATASET_ROOT / "val_data.json")
test_df = load_ndjson(DATASET_ROOT / "test_data.json")

all_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
all_df["label"] = all_df.get("class_label", all_df.get("label"))

print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df), "Total:", len(all_df))
all_df.head(3)

## 3) Class Distribution in Full Annotations

In [ ]:
class_counts = all_df["label"].value_counts().reindex(CLASS_NAMES, fill_value=0)

display(class_counts.to_frame("count"))

plt.figure(figsize=(13, 5))
sns.barplot(x=class_counts.index, y=class_counts.values, palette="viridis")
plt.xticks(rotation=45, ha="right")
plt.title("Class Distribution in Full Annotation Pool")
plt.ylabel("Image count")
plt.tight_layout()
plt.show()

## 4) Random Sample Images

In [ ]:
sample_df = all_df.sample(n=min(12, len(all_df)), random_state=42).reset_index(drop=True)

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for i, ax in enumerate(axes.flat):
    if i >= len(sample_df):
        ax.axis("off")
        continue

    row = sample_df.iloc[i]
    img_path = DATASET_ROOT / row["image_path"]
    try:
        img = Image.open(img_path).convert("RGB")
        ax.imshow(img)
        ax.set_title(str(row["label"]), fontsize=9)
    except Exception:
        ax.text(0.5, 0.5, "Unreadable", ha="center", va="center")
    ax.axis("off")

plt.tight_layout()
plt.show()

## 5) Image Resolution Statistics

In [ ]:
# To keep notebook fast, inspect a sample of images.
size_sample = all_df.sample(n=min(1000, len(all_df)), random_state=7)
widths, heights = [], []

for rel in size_sample["image_path"].tolist():
    p = DATASET_ROOT / rel
    try:
        with Image.open(p) as img:
            w, h = img.size
            widths.append(w)
            heights.append(h)
    except Exception:
        continue

size_df = pd.DataFrame({"width": widths, "height": heights})
size_df["aspect_ratio"] = size_df["width"] / size_df["height"]

display(size_df.describe().round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(size_df["width"], bins=30, ax=axes[0], color="steelblue")
axes[0].set_title("Width Distribution")
sns.histplot(size_df["height"], bins=30, ax=axes[1], color="darkorange")
axes[1].set_title("Height Distribution")
plt.tight_layout()
plt.show()

## 6) Verify Prepared Subset Counts

In [ ]:
def count_split(split: str) -> pd.Series:
    split_root = SUBSET_ROOT / split
    if not split_root.exists():
        return pd.Series(dtype="int64")

    counts = {}
    for cls in CLASS_NAMES:
        cls_dir = split_root / cls
        counts[cls] = len(list(cls_dir.glob("*"))) if cls_dir.exists() else 0
    return pd.Series(counts, name=f"{split}_count")

train_counts = count_split("train")
val_counts = count_split("val")
subset_counts = pd.concat([train_counts, val_counts], axis=1).fillna(0).astype(int)
subset_counts["total"] = subset_counts.sum(axis=1)

display(subset_counts)
print("Total subset images:", int(subset_counts["total"].sum()))

## 7) Load Training Outputs (Curves + Confusion Matrix)

In [ ]:
log_path = OUTPUTS_ROOT / "logs" / "training_log.csv"
curve_path = OUTPUTS_ROOT / "plots" / "training_curves.png"
cm_path = OUTPUTS_ROOT / "plots" / "confusion_matrix.png"

if log_path.exists():
    history = pd.read_csv(log_path)
    display(history.tail())
else:
    print("Training log not found:", log_path)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

if curve_path.exists():
    axes[0].imshow(Image.open(curve_path))
    axes[0].set_title("Saved Training Curves")
else:
    axes[0].text(0.5, 0.5, "training_curves.png not found", ha="center", va="center")
axes[0].axis("off")

if cm_path.exists():
    axes[1].imshow(Image.open(cm_path))
    axes[1].set_title("Saved Confusion Matrix")
else:
    axes[1].text(0.5, 0.5, "confusion_matrix.png not found", ha="center", va="center")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## 8) Run Summary (Real Dataset)

### Key Metrics
- Best validation accuracy: **63.08%**
- Top-5 validation accuracy: **93.69%**
- Best checkpoint epoch: **5**
- Model: **EfficientNet-B0** (ImageNet pretrained, transfer learning)

### Dataset Notes
- Target subset plan was 500 images per class across 15 classes.
- In the provided dataset export, `nehru_jacket` and `palazzo` had **0 available samples**.
- Effective subset used in training/validation: **13 classes** with available images.

### Training Notes
- Head-only fine-tuning (epochs 1-5) produced strong gains.
- Full-backbone fine-tuning on CPU became significantly slower after unfreezing.
- Final evaluation was performed using the best saved checkpoint.

### Artifacts
- Training log: `outputs/logs/training_log.csv`
- Training curves: `outputs/plots/training_curves.png`
- Confusion matrix: `outputs/plots/confusion_matrix.png`
- Sample predictions: `outputs/plots/sample_predictions.png`